
# Incident Analysis With LLM

Ноутбук:
1. Загружает таблицу в формате `Incident_analysis_march.ipynb`.
2. Выполняет минимальную предобработку через `IncidentRequestsPreprocessor` без `natasha`.
3. Запускает два LLM-эксперимента на одной и той же таблице.

Теперь каждый эксперимент оформлен отдельно:
- свой конфиг,
- свой prompt,
- свой запуск,
- свой `xlsx`-файл,
- свой `head()` и `value_counts()`.


In [ ]:

from functools import partial
from pathlib import Path
import json
import os
import re
import sys
from typing import Callable

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub.utils import enable_progress_bars

ROOT = Path.cwd()
if not (ROOT / "incident_requests").exists() and (ROOT.parent / "incident_requests").exists():
    ROOT = ROOT.parent

sys.path.append(str(ROOT))

from incident_requests import IncidentRequestsPreprocessor

pd.set_option("display.max_colwidth", None)
enable_progress_bars()

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"


In [ ]:

# Общий конфиг: данные и модель
INPUT_PATH = ROOT / "research/data" / "march_incidents.xlsx"
SHEET_NAME = 0

COLS2DROP = [
    "Источник",
    "Категория",
    "Комментарий к выполненным работам",
    "Статус заявки",
]

TEXT_COLUMN = "Описание"
TYPE_COLUMN = "Тип инцидента"
OTHER_LABEL = "Прочее"
EMPTY_LABEL = "Не определен"

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_CACHE_DIR = ROOT / ".cache" / "huggingface"
LOCAL_FILES_ONLY = False  # После первой успешной загрузки можно переключить в True.
DEVICE_MAP = "auto"
MAX_NEW_TOKENS = 96


In [ ]:

def load_table(path: Path, sheet_name=0) -> pd.DataFrame:
    suffix = path.suffix.lower()

    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path, sheet_name=sheet_name)

    if suffix == ".csv":
        return pd.read_csv(path)

    raise ValueError(f"Неподдерживаемый формат файла: {suffix}")


df = load_table(INPUT_PATH, sheet_name=SHEET_NAME)
processor = IncidentRequestsPreprocessor(
    df,
    columns_to_drop=COLS2DROP,
    detect_incident_type=False,
    use_natasha=False,
)
preprocessed_df = processor.preprocess()

print(preprocessed_df.dtypes)
display(preprocessed_df.head())


In [ ]:

def load_generation_model(
    model_name: str,
    cache_dir: Path | str | None = None,
    local_files_only: bool = False,
):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        cache_dir=cache_dir,
        local_files_only=local_files_only,
        trust_remote_code=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        cache_dir=cache_dir,
        local_files_only=local_files_only,
        torch_dtype=torch_dtype,
        device_map=DEVICE_MAP,
        trust_remote_code=True,
    )
    model.eval()
    return tokenizer, model


def parse_model_response(response_text: str, labels: list[str], other_label: str) -> str:
    match = re.search(r"\{.*\}", response_text, flags=re.S)
    if match:
        try:
            payload = json.loads(match.group(0))
            label = str(payload.get("incident_type", "")).strip()
            if label in labels:
                return label
        except json.JSONDecodeError:
            pass

    cleaned_text = response_text.strip().strip('"')
    if cleaned_text in labels:
        return cleaned_text

    for label in labels:
        if label.lower() in cleaned_text.lower():
            return label

    return other_label


def build_prompt(
    text: str,
    incident_types: list[str],
    prompt_instructions: str,
    other_label: str = OTHER_LABEL,
) -> str:
    labels = [*incident_types, other_label]
    options = "\n".join(f"- {label}" for label in labels)

    return f"""{prompt_instructions}

Доступные типы инцидентов:
{options}

Текст заявки:
{text}

Верни только JSON без пояснений:
{{"incident_type": "<один вариант из списка>"}}"""


def get_model_input_device(model) -> torch.device:
    return next(model.parameters()).device


def generate_incident_type(
    description: str,
    tokenizer,
    model,
    incident_types: list[str],
    prompt_builder: Callable[[str], str],
    other_label: str = OTHER_LABEL,
    empty_label: str = EMPTY_LABEL,
) -> str:
    if pd.isna(description) or not str(description).strip():
        return empty_label

    labels = [*incident_types, other_label]
    messages = [
        {
            "role": "system",
            "content": "Ты размечаешь заявки ЖКХ и возвращаешь только валидный JSON.",
        },
        {
            "role": "user",
            "content": prompt_builder(str(description)),
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer(prompt, return_tensors="pt")
    input_device = get_model_input_device(model)
    model_inputs = {key: value.to(input_device) for key, value in model_inputs.items()}

    with torch.inference_mode():
        generated = model.generate(
            **model_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = generated[0][model_inputs["input_ids"].shape[1]:]
    response_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return parse_model_response(response_text, labels, other_label)


def classify_incidents(
    dataframe: pd.DataFrame,
    tokenizer,
    model,
    text_column: str,
    incident_types: list[str],
    prompt_builder: Callable[[str], str],
    progress_label: str,
) -> pd.DataFrame:
    if text_column not in dataframe.columns:
        raise ValueError(f"Колонка '{text_column}' не найдена. Есть: {list(dataframe.columns)}")

    result_df = dataframe.copy()
    result_df[TYPE_COLUMN] = [
        generate_incident_type(description, tokenizer, model, incident_types, prompt_builder)
        for description in tqdm(result_df[text_column], total=len(result_df), desc=progress_label)
    ]
    return result_df


def save_result_table(dataframe: pd.DataFrame, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.suffix == ".csv":
        dataframe.to_csv(output_path, index=False)
    elif output_path.suffix in {".xlsx", ".xls"}:
        dataframe.to_excel(output_path, index=False)
    else:
        dataframe.to_parquet(output_path, index=False)


In [ ]:

tokenizer, model = load_generation_model(
    MODEL_NAME,
    cache_dir=MODEL_CACHE_DIR,
    local_files_only=LOCAL_FILES_ONLY,
)



## Эксперимент 1

Базовый сценарий с категориями `Стояк/Труба ГВС/ХВС`, плюс отдельная категория `Лифты`.


In [ ]:

# Конфиг эксперимента 1
EXPERIMENT_1_NAME = "base_plus_lifts"
EXPERIMENT_1_TYPES = [
    "Стояк ГВС",
    "Труба ГВС",
    "Стояк ХВС",
    "Труба ХВС",
    "Лифты",
]
EXPERIMENT_1_OUTPUT_PATH = ROOT / "research" / "incident_analysis_llm_base_plus_lifts.xlsx"
EXPERIMENT_1_PROMPT_INSTRUCTIONS = """Определи тип инцидента по тексту заявки.
Выбери ровно один вариант из списка.
Если заявка относится к лифту, застреванию или неисправности лифта, выбери \"Лифты\".
Если ни один вариант не подходит, выбери \"Прочее\"."""


In [ ]:

experiment_1_df = classify_incidents(
    preprocessed_df,
    tokenizer=tokenizer,
    model=model,
    text_column=TEXT_COLUMN,
    incident_types=EXPERIMENT_1_TYPES,
    prompt_builder=partial(
        build_prompt,
        incident_types=EXPERIMENT_1_TYPES,
        prompt_instructions=EXPERIMENT_1_PROMPT_INSTRUCTIONS,
    ),
    progress_label=EXPERIMENT_1_NAME,
)
save_result_table(experiment_1_df, EXPERIMENT_1_OUTPUT_PATH)

print(f"Файл сохранен: {EXPERIMENT_1_OUTPUT_PATH.resolve()}")
display(experiment_1_df[[TEXT_COLUMN, TYPE_COLUMN]].head())
display(
    experiment_1_df[TYPE_COLUMN]
    .value_counts(dropna=False)
    .rename_axis(TYPE_COLUMN)
    .reset_index(name="count")
)



## Эксперимент 2

Детальный сценарий с расширенной трубной таксономией и отдельной категорией `Лифты`.


In [ ]:

# Конфиг эксперимента 2
EXPERIMENT_2_NAME = "detailed_pipes_plus_lifts"
EXPERIMENT_2_TYPES = [
    "Анализ труб",
    "Утечка радиатора",
    "Утечка стояка",
    "Замена трубы ГВС",
    "Замена трубы ХВС",
    "Замена ХВС на уровне тех этажа",
    "Замена главных стояков ГВС",
    "Замена розлива",
    "Лифты",
]
EXPERIMENT_2_OUTPUT_PATH = ROOT / "research" / "incident_analysis_llm_detailed_pipes_plus_lifts.xlsx"
EXPERIMENT_2_PROMPT_INSTRUCTIONS = """Определи тип инцидента по тексту заявки ЖКХ.
Выбери ровно один вариант из списка.
Если в тексте несколько действий, выбери основное событие или основную работу.

Подсказки по классам:
- Анализ труб: обследование, диагностика, осмотр, анализ труб без явной замены как основного действия.
- Утечка радиатора: течь радиатора, батареи или похожего отопительного прибора.
- Утечка стояка: течь стояка.
- Замена трубы ГВС: замена трубы или трубопровода горячего водоснабжения.
- Замена трубы ХВС: замена трубы или трубопровода холодного водоснабжения.
- Замена ХВС на уровне тех этажа: замена ХВС на техэтаже или техническом этаже.
- Замена главных стояков ГВС: замена именно главных стояков ГВС.
- Замена розлива: замена розлива.
- Лифты: лифт не работает, застревание, неисправность лифта, остановка лифта.
- Прочее: если ни один вариант не подходит."""


In [ ]:

experiment_2_df = classify_incidents(
    preprocessed_df,
    tokenizer=tokenizer,
    model=model,
    text_column=TEXT_COLUMN,
    incident_types=EXPERIMENT_2_TYPES,
    prompt_builder=partial(
        build_prompt,
        incident_types=EXPERIMENT_2_TYPES,
        prompt_instructions=EXPERIMENT_2_PROMPT_INSTRUCTIONS,
    ),
    progress_label=EXPERIMENT_2_NAME,
)
save_result_table(experiment_2_df, EXPERIMENT_2_OUTPUT_PATH)

print(f"Файл сохранен: {EXPERIMENT_2_OUTPUT_PATH.resolve()}")
display(experiment_2_df[[TEXT_COLUMN, TYPE_COLUMN]].head())
display(
    experiment_2_df[TYPE_COLUMN]
    .value_counts(dropna=False)
    .rename_axis(TYPE_COLUMN)
    .reset_index(name="count")
)
